# orig

In [110]:
import sys
sys.path.append('/Users/brianlerner/Library/CloudStorage/OneDrive-Personal/Code/cerebra/src')
from datetime import datetime, timedelta
from pathlib import Path
from utils.io import load_yaml_file, load_json
from utils.general import to_snake_case
import pandas as pd
import itertools
import random
import numpy as np



In [122]:
def get_specs():
    
    profile = 'llm_v1'

    inputs_dir = Path.cwd().parent.parent /'data'/'sim' / 'profiles'/profile/'inputs'
    outputs_dir = Path.cwd().parent.parent /'data'/'sim' / 'profiles'/profile/'outputs'

    specs = []
    by_cat = {}
    for p in inputs_dir.glob("*.yaml"):
        if 'sim_params' in p.name:
            continue

        cat = p.stem
        cat_events = load_yaml_file(p)

        
        for event_info in cat_events:
            event_info["category"] = cat
            if 'calendar' in cat:
                event_info["scheduled"] = True
            specs.append(event_info)

        by_cat[cat] = cat_events
    
        
    return by_cat

# Function to parse days into a standardized format
def enforce_list(_):
    if isinstance(_, list):
        return _
    else:
        return [_]
    
def convert_date(date_string):
    try:
        return datetime.strptime(date_string, '%Y-%m-%d')
    except:
        print(date_string)
        raise

def convert_start_time(time_string):
    t = timedelta()
    parts = (p for p in ['hours','minutes','seconds'])
    for p in time_string.split(":"):
        t += timedelta(**{next(parts):float(p)})
    return t
    

def convert_end_time(duration, start_time):
    return start_time + timedelta(**duration)

def check_day(spec, date):

    check_counter = 0

    # check if the event occurs within the correct date range

    if 'start_date' in spec:
        if date<spec['start_date']:
            check_counter += 1
    if 'end_date' in spec:
        if date>spec['end_date']:
            check_counter += 1

    # check if event occurs on the specified day
    if days := spec.get('days'):
        if not date.strftime('%a') in days:
            check_counter += 1

    if check_counter == 0:
        return True
    else:
        return False
    
def get_sorted_pairs(day_group):
    sorted_combs = []
    for group_a, group_b in itertools.combinations(day_group, 2):
        # print(group_a)
        # print(group_b)
        time_a = group_a['start_time']
        time_b = group_b['start_time']
        if time_a < time_b:
            sorted_combs.append((group_a, group_b))
        else:
            sorted_combs.append((group_b, group_a))

        prec_a = group_a['start_time']
        prec_b = group_b['start_time']
    # now sort by precision
            
    


    # this can happen later if I want
    return sorted_combs

    

the first thing to do is to construct theevents that are scheduled
This means for events that have a set schedule, let's map those out.

In [123]:
profile_path = "/Users/brianlerner/Library/CloudStorage/OneDrive-Personal/Code/Cerebra/data/sim/profiles/llm_v1"
inputs_path = Path(profile_path) / "inputs"

In [175]:
combined_day_groups_meals[0][3]

{'name': 'Lunch',
 'start_time': datetime.timedelta(seconds=43200),
 'duration': datetime.timedelta(seconds=1800),
 'category': 'meals',
 'id': 'lunch',
 'end_time': datetime.timedelta(seconds=45000),
 'start_time_hrs': 12.0,
 'end_time_hrs': 12.5,
 'recipe': {'recipe_name': "Healthy Athlete's Turkey and Avocado Whole-Wheat Sandwich",
  'ingredients': [{'name': 'Bread, whole-wheat, commercially prepared',
    'grams': 60,
    'nutrients': [{'name': 'energy',
      'unit': 'kcal',
      'amount_per_100g': 254,
      'amount': 152.4}]},
   {'name': 'Turkey, ground, 93% lean, 7% fat, pan-broiled crumbles',
    'grams': 100,
    'nutrients': [{'name': 'energy',
      'unit': 'kcal',
      'amount_per_100g': 220,
      'amount': 220.0}]},
   {'name': 'Lettuce, romaine, green, raw',
    'grams': 20,
    'nutrients': [{'name': 'energy',
      'unit': 'kcal',
      'amount_per_100g': 20.8,
      'amount': 4.16}]},
   {'name': 'Tomatoes, grape, raw',
    'grams': 50,
    'nutrients': [{'name': 

In [230]:

# get the specs for scheduled events

specs = get_specs()

# set up date range for sim

sim_params_path = inputs_path / "sim_params.yaml"
sim_params = load_yaml_file(sim_params_path)
default_start_date = "2022-09-01"
default_end_date = "2022-12-31"
sim_start_date = convert_date(sim_params.get("start_date", default_start_date))
sim_end_date = convert_date(sim_params.get("end_date", default_end_date))

# get dates
# start_date = convert_date(sim_start_date)
# end_date = convert_date(sim_end_date)
total_days = (sim_end_date - sim_start_date).days
dates = [sim_start_date+timedelta(days=i) for i in range(total_days+1)]

# define whole day 
START_DAY_TIME = "00:00:00"
WHOLE_DAY_DURATION = {"days":1}
DEFAULT_DURATION = {"hours":1}

# MAKE SPECIAL ID FOR MEAL TYPE....


def format_dates(spec):
    # set up dates
    if not spec.get('start_date'):
        spec['start_date'] = sim_start_date
    else:
        spec['start_date'] = convert_date(spec['start_date'])

    if not spec.get('end_date'):
        spec['end_date'] = sim_end_date
    else:
        spec['end_date'] = convert_date(spec['end_date'])
    if spec['start_date']  > spec['end_date']:
        print("Start date comes before end date.")
        raise

def format_times(spec):
        

        # set up times
        if spec.get('delay'):
            if spec.get('all_day'):
                print('CONFLICT: delay w/ all_day')
                raise Exception
            if spec.get('start_time'):
                print('CONFLICT: all_day w/ start_time')
                raise Exception
            if spec.get('end_time'):
                print('CONFLICT: all_day w/ end_time')
                raise Exception
            elif not spec.get('duration'):
                spec['duration'] = DEFAULT_DURATION

            # format
            spec['delay'] = timedelta(**spec['delay'])
            spec['duration'] = timedelta(**spec['duration'])
        else:
            if spec.get('all_day'):
                # deal with conflicts
                if spec.get('start_time') or spec.get('duration'):
                    if spec.get('start_time'):
                        print('CONFLICT: all_day w/ start_time')
                        raise Exception
                    if spec.get('end_time'):
                        print('CONFLICT: all_day w/ end_time')
                        raise Exception
                    if spec.get('duration'):
                        print('CONFLICT: all_day w/ duration')
                        raise Exception
                    if spec.get('delay'):
                        print('CONFLICT: all_day w/ delay')
                        raise Exception
                # set times
                spec['start_time'] = START_DAY_TIME
                spec['duration'] = WHOLE_DAY_DURATION
            else:
                if not spec.get('start_time'):
                    print("Need start time")
                    print(spec)
                    raise Exception
                if not spec.get('duration'):
                    print("Need duration")
                    raise Exception
        
            spec['start_time'] = convert_start_time(spec['start_time'])
            spec["end_time"] = convert_end_time(spec["duration"], spec["start_time"])
            spec["start_time_hrs"] = spec["start_time"].total_seconds()/60**2
            spec["end_time_hrs"] = spec["end_time"].total_seconds()/60**2
            spec['duration'] = timedelta(**spec['duration'])
# return scheduled_events
        
def format_category(specs, category='scheduled'):


    process_dates = ['scheduled', 'sensations', 'unscheduled', 'scores']
    process_times = process_dates + ['meals']

    # check timing
    
    for spec in specs:

        # give id
        spec['id'] = to_snake_case(spec['name'])

        if category in process_dates:
            format_dates(spec)
        if category in process_times:
            format_times(spec)

        
def format_events(specs):

    for category, specs in specs.items():   
        format_category(specs, category=category)




def check_single_cause(cause, existing_events_for_day):

    cause_types = ['ingredient', 'standard']

    cause_type = cause.get('type', 'standard')
    if cause_type not in cause_types:
        print("cause type not recognized")
        raise Exception

    for existing_event in existing_events_for_day:
        if cause_type == 'ingredient':
            if recipe := existing_event.get('recipe'):
                ingredient_names = [i['name'] for i in recipe['ingredients']]
                if cause['name'] in ingredient_names:
                    return existing_event
        elif cause_type == 'standard':

            if existing_event['name'] == cause['name']:
                return existing_event
    return None # this means no cause was found

def check_causes(spec, existing_events_for_day):


    cause = spec['causes'][0] # not set up for multiple causes yet
    if existing_event := check_single_cause(cause, existing_events_for_day):
        if 'delay' in spec:
            spec["start_time"] = existing_event['start_time'] + spec['delay']
            spec["end_time"] = spec['start_time'] + spec['duration']
        return True
    else:
        return False
    

def make_day_groups(events, dates, existing_events=None):
    day_groups = []
    for i, date in enumerate(dates):

        day_group = []
        for spec in events:
            if not check_day(spec, date):
                continue
            if existing_events:
                if not 'causes' in spec:
                    continue
                if not check_causes(spec, existing_events[i]):
                    continue
            # deal with probability
            if prob := spec.get('prob'):
                if random.random() > prob:
                    day_group.append(spec)
            else:
                day_group.append(spec)
        day_groups.append(day_group)
    return day_groups

# first deal with precedence

def sort_precision(day_conflicts):
    def get_top_precedence(tup):
        s1, s2 = tup
        s1_precedence = s1.get("precedence", 5)
        s2_precedence = s2.get("precedence", 5)
        return min(s1_precedence, s2_precedence)

    day_conflicts.sort(key=get_top_precedence)
    return day_conflicts

def make_conflict_groups(day_groups):

    conflicts = []
    for dg in day_groups: 
        day_conflicts = []
        if len(dg) < 2:
            pass
        else:
            for s1, s2 in get_sorted_pairs(dg):
                id_1 = s1['id']
                id_2 = s2['id']
                if s1["end_time"] >= s2["start_time"] >= s1["start_time"]:
                    # to make sorting precision easier later
                    if s1.get('precedence',5) < 0 or s2.get('precedence',5) < 0:
                        continue
                    else:
                    # day_conflicts.append((id_1, id_2))
                        day_conflicts.append((s1, s2))
        conflicts.append(day_conflicts)

    return conflicts


def make_removal_groups(conflicts):
    removals = []
    for day_conflicts in conflicts:

        day_remove = []
        day_conflicts = sort_precision(day_conflicts)

        for s1,s2 in day_conflicts:

            id_1 = s1['id']
            id_2 = s2['id']
            s1_prec = s1.get("precedence", 5)
            s2_prec = s2.get("precedence", 5)

            # check if already removed
            existing_remove_ids = [s['id'] for s in day_remove]
            if id_1 in existing_remove_ids or id_2 in existing_remove_ids:
                continue

            if s1_prec<0 or s2_prec<0:
                continue
            elif s1_prec == s2_prec:
                print(f"Precedence conflict:{id_1}, {id_2}")

                # choose conflict randomly
                remove_spec = [s1, s1][round(random.random())]
            elif s1_prec > s2_prec:
                remove_spec = s2
            else:
                remove_spec = s1
            # remove_id = remove_spec['id']
            day_remove.append(remove_spec)
        removals.append(day_remove)
    return removals

def remove_conflicts(day_groups, removals):
    day_groups_final = []
    for r, dg in zip(removals, day_groups):
        # print*
        r_ids = [s['id'] for s in r]
        day_group = [s for s in dg if s['id'] not in r_ids]
        day_groups_final.append(day_group)
    return day_groups_final

def format_final_specs(day_groups):
    day_groups_final = []
    for date, dg in zip(dates, day_groups):
        # print(date)
        temp_dg = []
        for s in dg:
            temp_s = s.copy()
            temp_s['start_date'] = date
            # print(date)
            # s['start_date'] = date
            for k in ['days']:
                temp_s.pop(k, None)
            temp_dg.append(temp_s)

        day_groups_final.append(temp_dg)
    return day_groups_final

def combine_day_groups(dg1, dg2):
    combined_day_groups = []
    for d1, d2 in zip(dg1, dg2):
        combined_day_groups.append(d1+d2)
    return combined_day_groups

def deal_with_conflicts(day_groups):
    conflicts = make_conflict_groups(day_groups)
    removals = make_removal_groups(conflicts)
    return  remove_conflicts(day_groups, removals)

def determine_final_day_groups(events):
    day_groups = make_day_groups(events, dates)
    day_groups = deal_with_conflicts(day_groups)
    return day_groups
    
def add_recipes_to_meals(day_groups, recipes):
    for day_group in day_groups:
        for event in day_group:
            if event['category'] == 'meals':
                if not event['id'] in recipes:
                    # print(f"Recipe not found for {event['id']}")
                    continue
                    # raise Exception
                else:
                    event['recipe'] = random.choice(recipes[event['id']])


def adjust_score(effect, current_scores):

    def sample_score(effect):
        values = effect['vals']

        if 'probs' in effect:
            probabilities = effect['probs']
        else:
            probabilities = [1/len(values) for _ in values]
            
        return np.random.choice(values, p=probabilities)
    
    # get current score for effect
    cs = [cs for cs in current_scores if cs['name'] == effect['name']][0]

    # increment score
    cs['score'] += sample_score(effect)

    # adjust bounds
    if cs['score'] < cs['min']:
        cs['score'] = cs['min']
    elif cs['score'] > cs['max']:
        cs['score'] = cs['max']




def apply_effects(event, current_scores):
    score_names = [s['name'] for s in current_scores]
    for effect in event.get('effects', []):
        # check to make sure score name exists
        if effect['name'] not in score_names:
            print(f"Score name {effect['name']} not found")
            raise Exception
        adjust_score(effect, current_scores)

def make_scores(combined_day_groups_final):
    final_groups = []
    for dg in combined_day_groups_final:

        # set up current scores
        current_scores = [s.copy() for s in specs['scores']]
        for cs in current_scores:
            cs['score'] = cs['baseline']

        for event in dg:
                
            # apply event effects
            apply_effects(event, current_scores)


            # apply person effects
            if people := event.get('people'):
                for person in people:
                    # check to see if person exists
                    if person not in [p['name'] for p in specs['people']]:
                        print(f"{person} not found")
                        raise Exception
                    else:
                        # apply effects
                        person_spec = [p for p in specs['people'] if p['name'] == person][0]
                        apply_effects(person_spec, current_scores)

        final_groups.append(dg.copy() + current_scores)
    return final_groups
    
def get_full_df(day_groups_final):
    all_events = [s for specs in day_groups_final for s in specs]
    df_all = pd.DataFrame(all_events)
    df_all.start_date.unique()
    df_all.sort_values(['start_date', 'start_time_hrs'], inplace=True)
    return df_all

profile = 'llm_v1'

inputs_dir = Path.cwd().parent.parent /'data'/'sim' / 'profiles'/profile/'inputs'
outputs_dir = Path.cwd().parent.parent /'data'/'sim' / 'profiles'/profile/'outputs'

recipes_path = outputs_dir / 'intermediate' / 'recipes_final.json'
recipes = load_json(recipes_path)




# setup
specs = get_specs()
format_events(specs)

# scheduled events
day_groups_final_scheduled = determine_final_day_groups(specs['scheduled'])

# unscheduled events
day_groups_final_unscheduled = determine_final_day_groups(specs['unscheduled'])
combined_day_groups = combine_day_groups(day_groups_final_scheduled, day_groups_final_unscheduled)
day_groups_final = deal_with_conflicts(combined_day_groups)
    
# add meals
day_groups_meals = make_day_groups(specs['meals'], dates)
add_recipes_to_meals(day_groups_meals, recipes)
combined_day_groups_meals = combine_day_groups(day_groups_final, day_groups_meals)

# add sensations
day_groups_sensations = make_day_groups(specs['sensations'], dates, existing_events=combined_day_groups_meals)
combined_day_groups = combine_day_groups(combined_day_groups_meals, day_groups_sensations)

# create scores
final_groups = make_scores(combined_day_groups)

final_groups = format_final_specs(final_groups)
df = get_full_df(final_groups)
# display(df)
# df.groupby('start_date').filter(lambda x: x.id.str.contains('head').any())
# # do this at the end





Precedence conflict:all_day_party, movie_night
Precedence conflict:all_day_party, movie_night
Precedence conflict:all_day_party, movie_night
Precedence conflict:all_day_party, movie_night


In [235]:
df.set_index('start_date')

,name,group,location,description,start_time,duration,category,id,end_date,end_time,...,score,type,enjoyed,groups,causes,delay,prob,intensity,effects,people
start_date,,,,,,,,,,,,,,,,,,,,,
2022-09-01,Computer Programming HW due,homeworks,None,HW.,0 days 00:00:00,1 days 00:00:00,scheduled,computer_programming_hw_due,2022-12-15,1 days 00:00:00,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
2022-09-01,mood,NaN,NaN,NaN,0 days 00:00:00,1 days 00:00:00,scores,mood,2023-04-30,1 days 00:00:00,...,5.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
2022-09-01,energy,NaN,NaN,NaN,0 days 00:00:00,1 days 00:00:00,scores,energy,2023-04-30,1 days 00:00:00,...,5.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
2022-09-01,stress,NaN,NaN,NaN,0 days 00:00:00,1 days 00:00:00,scores,stress,2023-04-30,1 days 00:00:00,...,5.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
2022-09-01,Computer Programming,NaN,Hudson Hall,Class.,0 days 08:30:00,0 days 01:15:00,scheduled,computer_programming,2022-12-15,0 days 09:45:00,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-04-30,stress,NaN,NaN,NaN,0 days 00:00:00,1 days 00:00:00,scores,stress,2023-04-30,1 days 00:00:00,...,5.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
2023-04-30,Breakfast,NaN,NaN,NaN,0 days 07:00:00,0 days 00:30:00,meals,breakfast,NaT,0 days 07:30:00,...,NaN,NaN,NaN,NaN,NaN,NaT,0.5,NaN,NaN,NaN
2023-04-30,Lunch,NaN,NaN,NaN,0 days 12:00:00,0 days 00:30:00,meals,lunch,NaT,0 days 12:30:00,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN


In [212]:
specs['people']

[{'name': 'Bobby',
  'effects': [{'mood': None, 'vals': [-1, 0], 'probs': [0.8, 0.2]}],
  'description': 'friend',
  'category': 'people',
  'id': 'bobby'},
 {'name': 'Tanya',
  'effects': [{'mood': None, 'vals': [-1, 0, 1], 'probs': [0.1, 0.3, 0.6]}],
  'description': 'girlfriend',
  'category': 'people',
  'id': 'tanya'}]

After the scheduled events, here's what I need to enter, in order:
- unscheduled
- sensations
- meals

When I create the day_groups
- I check if the day is correct
- Add event based on prob

When I make the conflict groups
- I find conflicts based on time
- check if any -1 precedences

When I make the removal groups
- I use precdence to determine what to remove

When I remove conflicts
- I simply remove the conflicts


So, for adding the unscheduled conflicts on to what I have already
- I need to get the final day groups 
- AND then I need to remove items if they conflict with the scheduled events

Now that I've mapped out the scheduled events, it's time to put in the hobbies.
What's different between the hobbies and the scheduled events. Well, for one, the hobbies cannot happen if they conflict with the scheduled events. Thus, the right thing to do is to probably sample probablisticslly to determine the start date (if a prob is attached), and if there is a conflict, then it won't happen.

Here's the plan, to keep things simple:
- for each hobby, determine a datetime if no conflict with scheduled events
- then check if any conflicts with other hobbies
- if no conflicts, then add to schedule

In [333]:
len(dates), len(conflicts), len(removals), len(day_groups_final)

(242, 242, 242, 242)

In [ ]:



def construct_schedule(profile='llm_v1'):


    sim_params, calendar_events, sensations, hobbies = load_input_yaml(profile=profile)
    # Extract start and end dates from sim_params
    start_date = datetime.datetime.strptime(sim_params['start_date'], '%Y-%m-%d')
    end_date = datetime.datetime.strptime(sim_params['end_date'], '%Y-%m-%d')

    # Initialize an empty schedule
    total_schedule = []

    # Generate schedule for each day within the date range
    current_date = start_date
    while current_date <= end_date:

        



